In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pymilvus import MilvusClient

client = MilvusClient("milvus_diseases.db")
client.list_collections()

In [ ]:
from pymilvus.model.dense import SentenceTransformerEmbeddingFunction

# default is all-MiniLM-L6-v2 [384 dim]
embedding_fn = SentenceTransformerEmbeddingFunction(model_name='all-mpnet-base-v2')
embedding_fn.model_name


#### Load the data to embed

In [4]:
import pandas as pd

In [ ]:
# disease_syn_to_code = pd.read_csv('disease_syn_2_code.tsv', sep='\t')
# terms = pd.read_csv("code2term.tsv", sep="\t")
# terms = terms.rename(columns={"Preferred Term": "Term"})
dictionary_file = 'disease2code_lg_legacy (1).tsv'
terms = pd.read_csv(dictionary_file, sep='\t')
# terms['Code'] = [None]*len(terms)
# terms["Term"] = terms["Term"].apply(lambda t: t.lower())

terms, terms.duplicated().any()

In [ ]:
documents = terms["Term"].to_list()
metadatas = [{"code": c} for c in terms["Code"]]
ids = [f"id{i}" for i in terms.index]
len(ids)

In [ ]:
?embedding_fn

#### Produce and normalize the vectors

In [ ]:
vectors = embedding_fn.encode_documents(documents)

In [ ]:
(len(vectors), len(vectors[0]))

In [ ]:
import numpy as np
magnitudes = np.array([np.linalg.norm(v) for v in vectors])
norm_vectors = np.array([v / m for v, m in zip(vectors, magnitudes)])
print(np.allclose(magnitudes, [1]*len(magnitudes)))
print(np.allclose(vectors, norm_vectors))

#### Drop a collection

In [ ]:
client.drop_collection(collection_name)

#### Prepare and insert the vector data

In [11]:
collection_name = "disease2code_lg_legacy"
if client.has_collection(collection_name):
    print(client.get_collection_stats(collection_name))

In [12]:
data = [
    {
        "text": documents[i],
        # "vector_ip": vectors[i],
        "vector": vectors[i],
        "code": metadatas[i]["code"],
        "id": i,
    }
    for i in range(len(documents))
]

In [13]:
from pymilvus import DataType

schema = MilvusClient.create_schema(
    auto_id=False,
    enable_dynamic_field=True,
)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
# schema.add_field(
#     field_name="vector_ip", datatype=DataType.FLOAT_VECTOR, dim=embedding_fn.dim
# )
schema.add_field(
    field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=embedding_fn.dim
)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=1024)
schema.add_field(field_name="code", datatype=DataType.VARCHAR, max_length=1024)
client.create_collection(
    collection_name,
    schema=schema,
)

index_params = MilvusClient.prepare_index_params()
# See this API doc https://milvus.io/api-reference/pymilvus/v2.4.x/MilvusClient/Management/add_index.md
#
# See this guide for more info https://milvus.io/docs/index.md
# IVF_FLAT good for high recall and high-speed query (it is quantization based)
# index_params.add_index(
#     field_name="vector_ip",
#     index_type="AUTOINDEX",
#     metric_type="IP",
# )
index_params.add_index(
    field_name="vector",
    index_type="AUTOINDEX",
    metric_type="COSINE",
    params={"M": 4, "efConstruction": 400, "ef": 1000},
)
# COSINE and IP will produce the same distance scores for normalized embeddings,
# which are produced automatically by the SentenceTransformers embedding model.
client.create_index(
    collection_name=collection_name,
    index_params=index_params,
)

In [ ]:
client.describe_index(collection_name, index_name="vector")

In [ ]:
res = client.insert(collection_name=collection_name, data=data)
print(res['insert_count'])

> This may take around 5 min x # of indexes

#### Load the vector database collection

In [16]:
client.load_collection(collection_name=collection_name)

#### Run some test cases

In [ ]:
text = 'Stage IIB Lung Cancer'
query_vectors = embedding_fn.encode_queries([text])
res = client.search(
    collection_name=collection_name,
    anns_field="vector",
    data=query_vectors,
    limit=3,
    output_fields=["text",  "code"],
    search_params={"metric_type": "COSINE"},
)
e1 = res[0][0]['entity']
e2 = res[0][1]['entity']
e3 = res[0][2]['entity']
print(e1)
print(e2)
print(e3)